# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

## Paso 0. Carga y limpieza

El enunciado pide limpiar, formatear y estructurar antes de analizar. Sigo el mismo orden que en el lab de limpieza.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

(10910, 26)


,Unnamed: 0,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,Income,Location Code,Marital Status,Monthly Premium Auto,Months Since Last Claim,Months Since Policy Inception,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size,Vehicle Type
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,48029,Suburban,Married,61,7.0,52,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,0,Suburban,Single,64,3.0,26,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,22139,Suburban,Single,100,34.0,31,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,49078,Suburban,Single,97,10.0,3,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,23675,Suburban,Married,117,NaN,31,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN


**Nombres de columna:** minúsculas y `_`, como siempre. Elimino `unnamed:_0`: es el índice de cuando se guardó el archivo, no un dato del cliente.

In [2]:
df.columns = df.columns.str.lower().str.replace(" ", "_")
df = df.drop(columns=["unnamed:_0"])
df.columns

Index(['customer', 'state', 'customer_lifetime_value', 'response', 'coverage',
       'education', 'effective_to_date', 'employmentstatus', 'gender',
       'income', 'location_code', 'marital_status', 'monthly_premium_auto',
       'months_since_last_claim', 'months_since_policy_inception',
       'number_of_open_complaints', 'number_of_policies', 'policy_type',
       'policy', 'renew_offer_type', 'sales_channel', 'total_claim_amount',
       'vehicle_class', 'vehicle_size', 'vehicle_type'],
      dtype='object')

**Nulos:** los cuento y miro si coinciden en las mismas filas.

In [3]:
nulls = df.isnull().sum()
print(nulls[nulls > 0], "\n")

print("Rows missing both state and response:", (df["state"].isnull() & df["response"].isnull()).sum())
print("Values of vehicle_type:", df["vehicle_type"].unique())

state                         631
response                      631
months_since_last_claim       633
number_of_open_complaints     633
vehicle_class                 622
vehicle_size                  622
vehicle_type                 5482
dtype: int64 

Rows missing both state and response: 631
Values of vehicle_type: [nan 'A']


**Estrategia:**

| Columna | Nulos | Decisión | Por qué |
|---|---|---|---|
| `vehicle_type` | 5482 (≈ 50 %) | **eliminar la columna** | falta en la mitad de las filas y solo tiene un valor (`A`), así que no distingue a nadie |
| `state`, `response` | 631, en las mismas filas | **rellenar con `"Unknown"`** | el resto de datos de esas filas sirve para otras preguntas. No invento un estado ni una respuesta; cuando una pregunta dependa de ellos, excluyo `Unknown` |
| `vehicle_class`, `vehicle_size` | 622 | **rellenar con `"Unknown"`** | mismo razonamiento |
| `months_since_last_claim`, `number_of_open_complaints` | 633 | **rellenar con la mediana** | son numéricas; la mediana no se deja arrastrar por valores extremos |

**Por qué un bucle `for` con `fillna`:** son varias columnas con la misma regla; el bucle evita repetir la misma línea cuatro veces.

In [4]:
df = df.drop(columns=["vehicle_type"])

for column in ["state", "response", "vehicle_class", "vehicle_size"]:
    df[column] = df[column].fillna("Unknown")

for column in ["months_since_last_claim", "number_of_open_complaints"]:
    df[column] = df[column].fillna(df[column].median())

print("Nulls left:", df.isnull().sum().sum())

Nulls left: 0


**Formato de fechas:** `effective_to_date` viene como texto (`2/18/11`). `pd.to_datetime` con `format="%m/%d/%y"` (mes/día/año de dos cifras) la convierte en fecha. Después creo la columna `month` con `.dt.month`, que necesito para las preguntas 5 y 6.

**Por qué indicar el `format`:** sin él, pandas tiene que adivinar si `2/10/11` es 10 de febrero o 2 de octubre. Con el formato explícito no hay ambigüedad.

In [5]:
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], format="%m/%d/%y")
df["month"] = df["effective_to_date"].dt.month
print(df["month"].value_counts())

month
1    5818
2    5092
Name: count, dtype: int64


**Duplicados:** hay filas idénticas y también clientes repetidos.

In [6]:
print("Identical rows:", df.duplicated().sum())
print("Repeated customer IDs:", df["customer"].duplicated().sum())

Identical rows: 70
Repeated customer IDs: 1776


Elimino solo las **filas idénticas**: son registros copiados. Los clientes repetidos con datos distintos los mantengo, porque no puedo saber cuál de sus filas es la buena y pueden corresponder a registros distintos del mismo cliente.

In [7]:
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)

(10840, 25)


1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

**Qué hago:** filtro con dos condiciones unidas por `&`, cada una entre paréntesis. `.copy()` crea un DataFrame independiente, para que modificarlo después no afecte a `df`.

In [8]:
low_claim_yes = df.loc[(df["total_claim_amount"] < 1000) & (df["response"] == "Yes")].copy()
print(low_claim_yes.shape)
low_claim_yes.head()

(1390, 25)


,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,income,location_code,marital_status,monthly_premium_auto,months_since_last_claim,months_since_policy_inception,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,month
3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,49078,Suburban,Single,97,10.0,3,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,1
8,FM55990,California,5989.773931,Yes,Premium,College,2011-01-19,Employed,M,66839,Suburban,Single,154,33.0,24,0.0,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,1
15,CW49887,California,4626.801093,Yes,Basic,Master,2011-01-16,Employed,F,79487,Suburban,Divorced,114,20.0,87,0.0,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,1
19,NJ54277,California,3746.751625,Yes,Extended,College,2011-02-26,Employed,F,41479,Rural,Married,94,14.0,38,1.0,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,2
27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2011-02-28,Employed,F,63774,Rural,Divorced,111,18.0,63,0.0,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,2


2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

**Qué hago:** me quedo con los clientes que respondieron `Yes` y hago una `pivot_table` con `policy_type` en filas, `gender` en columnas y la media de tres medidas: prima mensual, CLV y reclamación total.

**Por qué las tres a la vez:** la pregunta es sobre rentabilidad. Lo que ingresa la aseguradora (prima y CLV) solo tiene sentido comparado con lo que paga (reclamaciones).

In [9]:
responders = df.loc[df["response"] == "Yes"]

responders_summary = responders.pivot_table(
    index="policy_type",
    columns="gender",
    values=["monthly_premium_auto", "customer_lifetime_value", "total_claim_amount"],
    aggfunc="mean",
).round(2)
responders_summary

customer_lifetime_value           ... total_claim_amount        
gender                               F        M  ...                  F       M
policy_type                                      ...                           
Corporate Auto                 7712.63  7924.17  ...             433.74  409.34
Personal Auto                  8345.57  7438.52  ...             452.25  457.29
Special Auto                   7691.58  8247.09  ...             453.28  429.53

[3 rows x 6 columns]

Para comparar mejor calculo, para cada segmento, cuánto se reclama por cada unidad de CLV (`total_claim_amount / customer_lifetime_value`). Cuanto más bajo, más rentable es el segmento.

In [10]:
segments = responders.groupby(["policy_type", "gender"])[
    ["customer_lifetime_value", "monthly_premium_auto", "total_claim_amount"]
].mean()
segments = segments.round(2)
segments["claim_per_clv"] = (segments["total_claim_amount"] / segments["customer_lifetime_value"]).round(4)
segments.sort_values("claim_per_clv")

,,customer_lifetime_value,monthly_premium_auto,total_claim_amount,claim_per_clv
policy_type,gender,,,,
Corporate Auto,M,7924.17,92.03,409.34,0.0517
Special Auto,M,8247.09,86.34,429.53,0.0521
Personal Auto,F,8345.57,98.88,452.25,0.0542
Corporate Auto,F,7712.63,94.30,433.74,0.0562
Special Auto,F,7691.58,92.31,453.28,0.0589
Personal Auto,M,7438.52,91.09,457.29,0.0615


**Conclusiones:**

- **Más rentables / menor riesgo:** según la ratio reclamación/CLV, los hombres con Corporate Auto (la reclamación media más baja, ≈ 409), los hombres con Special Auto (CLV alto, ≈ 8247) y las mujeres con Personal Auto (el CLV más alto, ≈ 8346, con reclamación en la media).
- **Menos rentables:** los hombres con Personal Auto: tienen el CLV más bajo (≈ 7439) y la reclamación más alta (≈ 457).
- Las mujeres pagan algo más de prima mensual que los hombres en los tres tipos de póliza.
- Como Personal Auto es, con diferencia, el tipo con más clientes, la diferencia entre hombres y mujeres dentro de él es la que más pesa en el resultado de la empresa.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

**Qué hago:** `value_counts()` cuenta clientes por estado. Filtro la Series con una condición (`> 500`), igual que se filtra un DataFrame. `known_states` (sin `Unknown`) viene de la pregunta anterior.

In [11]:
customers_by_state = df.loc[df["state"] != "Unknown", "state"].value_counts()
print(customers_by_state, "\n")

big_states = customers_by_state[customers_by_state > 500]
big_states

state
California    3539
Oregon        2892
Arizona       1928
Nevada         987
Washington     882
Name: count, dtype: int64 



state
California    3539
Oregon        2892
Arizona       1928
Nevada         987
Washington     882
Name: count, dtype: int64

**Resultado:** los 5 estados superan los 500 clientes, así que el filtro no deja ninguno fuera. California (≈ 3500) y Oregon (≈ 2900) son los mercados principales; Washington es el más pequeño, pero aún así supera los 800.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

**Qué hago:** `groupby(["education", "gender"])` y `agg()` con una lista de funciones, que calcula las tres estadísticas de una vez.

**Por qué `agg` con lista y no tres `groupby` separados:** da una sola tabla con las tres columnas, lista para comparar.

In [12]:
clv_stats = df.groupby(["education", "gender"])["customer_lifetime_value"].agg(["max", "min", "median"]).round(2)
clv_stats

max      min   median
education            gender                            
Bachelor             F       73225.96  1904.00  5632.61
                     M       67907.27  1898.01  5548.03
College              F       61850.19  1898.68  5623.61
                     M       61134.68  1918.12  6002.61
Doctor               F       44856.11  2395.57  5332.46
                     M       32677.34  2267.60  5577.67
High School or Below F       55277.45  2144.92  6044.02
                     M       83325.38  1940.98  6285.31
Master               F       51016.07  2417.78  5717.22
                     M       50568.26  2272.31  5577.24

**Conclusiones:**

- **Mínimos:** muy parecidos en todos los grupos (≈ 1900 a 2400). Hay un "suelo" de valor por cliente.
- **Medianas:** también parecidas (≈ 5300 a 6300). La más alta es la de hombres con High School or Below; la más baja, la de mujeres con Doctor.
- **Máximos:** es donde hay diferencias. En los grupos grandes (High School or Below, Bachelor, College) aparecen clientes de más de 60 000; en Doctor el máximo no llega a 45 000. Con más clientes es más fácil que aparezca un valor extremo.
- Máximo muy lejos de la mediana en todos los grupos: la distribución del CLV es muy asimétrica, con pocos clientes muy valiosos. Por eso uso la mediana y no la media para describir al cliente típico.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

**Qué hago:** `pivot_table` con `state` en filas, `month` en columnas y `aggfunc="count"` sobre `customer`. Cada fila del dataset es una póliza, así que contar filas es contar pólizas.

**Por qué `count` y no `sum`:** no quiero sumar un valor, sino contar cuántas pólizas hay en cada cruce estado × mes.

Como en la pregunta 3, dejo fuera `Unknown`, que no es un estado.

In [13]:
known_states = df.loc[df["state"] != "Unknown"]
policies_state_month = known_states.pivot_table(index="state", columns="month", values="customer", aggfunc="count")
policies_state_month

month,1,2
state,,
Arizona,1005,923
California,1909,1630
Nevada,547,440
Oregon,1556,1336
Washington,460,422


En todos los estados se vendieron más pólizas en enero (mes 1) que en febrero (mes 2). El dataset solo cubre estos dos meses.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

**Qué hago, siguiendo las pistas:**

1. Agrupo por estado y mes y cuento pólizas (`size()` cuenta filas por grupo).
2. Calculo el total por estado, lo ordeno de mayor a menor y me quedo con los 3 primeros con `nlargest(3)`.
3. Filtro el resultado del paso 1 con `isin()` para quedarme solo con esos 3 estados.
4. Lo muestro con los meses como columnas con `unstack()`, que pasa el nivel `month` del índice a columnas.

`known_states` (sin `Unknown`) viene de la pregunta anterior.

In [14]:
policies_by_state_month = known_states.groupby(["state", "month"]).size().rename("policies_sold")

top_3_states = policies_by_state_month.groupby("state").sum().nlargest(3).index
print("Top 3 states:", list(top_3_states))

top_3 = policies_by_state_month[policies_by_state_month.index.get_level_values("state").isin(top_3_states)]
top_3_table = top_3.unstack("month").loc[top_3_states]
top_3_table

Top 3 states: ['California', 'Oregon', 'Arizona']


month,1,2
state,,
California,1909,1630
Oregon,1556,1336
Arizona,1005,923


7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

**Qué hago:**

1. `pd.crosstab(..., normalize="index")` da, para cada canal, la proporción de `Yes` y de `No`. Excluyo las respuestas `Unknown`, porque no sé si ese cliente respondió.
2. `melt()` hace lo contrario de `pivot`: pasa las columnas `No` y `Yes` a filas, así cada fila es un canal y un tipo de respuesta (formato largo).
3. Me quedo con las filas de `Yes`: esa proporción es la tasa de respuesta.

**Por qué `melt` aquí:** el enunciado lo pide para practicar el paso de formato ancho a largo. El formato largo es el que esperan muchas funciones de gráficos (como las de seaborn), que se verán más adelante.

In [15]:
known_response = df.loc[df["response"] != "Unknown"]

response_wide = pd.crosstab(known_response["sales_channel"], known_response["response"], normalize="index").reset_index()
response_long = response_wide.melt(id_vars="sales_channel", var_name="response", value_name="rate")

response_rate = (
    response_long.loc[response_long["response"] == "Yes", ["sales_channel", "rate"]]
    .sort_values("rate", ascending=False)
    .reset_index(drop=True)
)
response_rate["rate"] = response_rate["rate"].round(3)
response_rate

,sales_channel,rate
0,Agent,0.190
1,Web,0.118
2,Branch,0.114
3,Call Center,0.110


**Conclusión:** el canal **Agent** tiene una tasa de respuesta claramente mayor (≈ 19 %) que Branch, Web y Call Center (≈ 11 % a 12 %). El trato directo con un agente parece convertir mejor. Como Agent es además el canal con más clientes y más ingresos (lab anterior), es el candidato natural para concentrar las próximas campañas.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9